In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
!pip install catboost
from catboost import CatBoostClassifier

# Load dataset
df = pd.read_csv("lithium-ion batteries.csv")

# -----------------------------
# FIX DATA LEAKAGE (IMPORTANT)
# -----------------------------
X = df.drop(columns=["Crystal System", "Materials Id", "Formula", "Spacegroup"])
y = df["Crystal System"]

# Encode target
y = LabelEncoder().fit_transform(y)

# Convert bandstructure if exists
if "Has Bandstructure" in X.columns:
    X["Has Bandstructure"] = X["Has Bandstructure"].replace({"True":1,"False":0})

# -----------------------------
# K-FOLD
# -----------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# MODELS
# -----------------------------
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000))
    ]),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier())
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True))
    ]),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(eval_metric='mlogloss'),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# -----------------------------
# METRICS
# -----------------------------
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc_ovr"
}

results = []

for name, model in models.items():
    scores = cross_validate(model, X, y, cv=kfold, scoring=scoring)

    results.append({
        "Model": name,
        "Accuracy": np.mean(scores["test_accuracy"]),
        "Precision": np.mean(scores["test_precision"]),
        "Recall": np.mean(scores["test_recall"]),
        "F1": np.mean(scores["test_f1"]),
        "ROC-AUC": np.mean(scores["test_roc_auc"])
    })

results_df = pd.DataFrame(results)
results_df.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.4748,0.4411,0.4286,0.4240,0.6419
1,Decision Tree,0.5692,0.5596,0.5547,0.5541,0.6662
2,Random Forest,0.6251,0.6476,0.6018,0.6093,0.8000
3,KNN,0.4986,0.4496,0.4396,0.4207,0.6752
4,SVM,0.5221,0.4796,0.4511,0.4208,0.7052
5,Gradient Boosting,0.6223,0.6373,0.5993,0.6053,0.7849
6,XGBoost,0.5808,0.5881,0.5568,0.5612,0.7674
7,Naive Bayes,0.3893,0.3943,0.3879,0.3771,0.6094
8,AdaBoost,0.4926,0.5203,0.4540,0.4525,0.6957
9,CatBoost,0.5662,0.5650,0.5448,0.5482,0.7677


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Load dataset
df = pd.read_csv("lithium-ion batteries.csv")

# -----------------------------
# REMOVE LEAKAGE
# -----------------------------
X = df.drop(columns=["Crystal System", "Materials Id", "Formula", "Spacegroup"])
y = LabelEncoder().fit_transform(df["Crystal System"])

if "Has Bandstructure" in X.columns:
    X["Has Bandstructure"] = X["Has Bandstructure"].replace({"True":1,"False":0})

# -----------------------------
# K-FOLD
# -----------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
rf_pipeline = ImbPipeline([
    ('smote', SMOTE()),
    ('model', RandomForestClassifier())
])

rf_params = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=kfold, scoring='accuracy')
rf_grid.fit(X, y)

print("Best RF:", rf_grid.best_params_)
print("RF Accuracy:", rf_grid.best_score_)

Best RF: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 300}
RF Accuracy: 0.6339771729587358


In [ ]:
rf_pipeline = ImbPipeline([
    ('smote', SMOTE()),
    ('model', RandomForestClassifier())
])

rf_params = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=kfold, scoring='accuracy')
rf_grid.fit(X, y)

print("Best RF:", rf_grid.best_params_)
print("RF Accuracy:", rf_grid.best_score_)

Best RF: {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 200}
RF Accuracy: 0.6339771729587358


In [ ]:
cat_model = CatBoostClassifier(verbose=0)

cat_params = {
    'iterations': [100, 200],
    'depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1]
}

cat_grid = GridSearchCV(cat_model, cat_params, cv=kfold, scoring='accuracy')
cat_grid.fit(X, y)

print("Best CatBoost:", cat_grid.best_params_)
print("CatBoost Accuracy:", cat_grid.best_score_)

Best CatBoost: {'depth': 4, 'iterations': 200, 'learning_rate': 0.1}
CatBoost Accuracy: 0.5780070237050043


In [ ]:
gb_model = GradientBoostingClassifier()

gb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

gb_grid = GridSearchCV(gb_model, gb_params, cv=kfold, scoring='accuracy')
gb_grid.fit(X, y)

print("Best GB:", gb_grid.best_params_)
print("GB Accuracy:", gb_grid.best_score_)

Best GB: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
GB Accuracy: 0.6369622475856014


In [ ]:
xgb_pipeline = ImbPipeline([
    ('smote', SMOTE()),
    ('model', XGBClassifier(eval_metric='mlogloss'))
])

xgb_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.05, 0.1]
}

xgb_grid = GridSearchCV(xgb_pipeline, xgb_params, cv=kfold, scoring='accuracy')
xgb_grid.fit(X, y)

print("Best XGB:", xgb_grid.best_params_)
print("XGB Accuracy:", xgb_grid.best_score_)

Best XGB: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 200}
XGB Accuracy: 0.6223880597014926


In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import make_scorer
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv("lithium-ion batteries.csv")

# -----------------------------
# Parse formula into element counts
# -----------------------------
def parse_formula_counts(formula):
    tokens = re.findall(r'([A-Z][a-z]?)(\d*)', str(formula))
    counts = {}
    for el, num in tokens:
        counts[el] = counts.get(el, 0) + (int(num) if num else 1)
    return counts

elements = ["Li", "Co", "Fe", "Mn", "Si", "O"]

for el in elements:
    df[el] = df["Formula"].apply(lambda x: parse_formula_counts(x).get(el, 0))

df["TM_total"] = df[["Co", "Fe", "Mn"]].sum(axis=1)
df["Li_to_Si"] = df["Li"] / df["Si"].replace(0, np.nan)
df["TM_to_Si"] = df["TM_total"] / df["Si"].replace(0, np.nan)
df["O_to_Si"] = df["O"] / df["Si"].replace(0, np.nan)
df["Li_to_TM"] = df["Li"] / df["TM_total"].replace(0, np.nan)

# -----------------------------
# Target
# -----------------------------
y = df["Crystal System"]
y = LabelEncoder().fit_transform(y) # Add this line to encode target variable

# -----------------------------
# Features
# Keep Formula-derived features, but remove Spacegroup
# -----------------------------
feature_cols = [
    "Formation Energy (eV)",
    "E Above Hull (eV)",
    "Band Gap (eV)",
    "Nsites",
    "Density (gm/cc)",
    "Volume",
    "Has Bandstructure",
    "Li", "Co", "Fe", "Mn", "Si", "O",
    "TM_total", "Li_to_Si", "TM_to_Si", "O_to_Si", "Li_to_TM"
]

X = df[feature_cols].copy()

X["Has Bandstructure"] = X["Has Bandstructure"].replace({
    "TRUE": 1, "FALSE": 0, True: 1, False: 0
}).astype(float)

# -----------------------------
# Models
# -----------------------------
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=10,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        random_state=42
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        loss_function="MultiClass",
        verbose=False,
        random_state=42
    )
}

# -----------------------------
# CV
# -----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc_ovr"
}

rows = []

for name, model in models.items():
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=1)

    rows.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC_AUC": scores["test_roc_auc"].mean()
    })

results = pd.DataFrame(rows).sort_values("Accuracy", ascending=False)
print(results.round(4))

/tmp/ipykernel_6687/3121665976.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X["Has Bandstructure"] = X["Has Bandstructure"].replace({


               Model  Accuracy  Precision  Recall      F1  ROC_AUC
2            XGBoost    0.6724     0.7018  0.6559  0.6653   0.8292
0      Random Forest    0.6576     0.6698  0.6606  0.6600   0.8281
1  Gradient Boosting    0.6546     0.6723  0.6386  0.6397   0.8175
3           CatBoost    0.6457     0.6658  0.6344  0.6396   0.8217


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

# Load dataset
df = pd.read_csv("lithium-ion batteries.csv")

# -----------------------------
# REMOVE LEAKAGE
# -----------------------------
X = df.drop(columns=["Crystal System", "Materials Id", "Formula", "Spacegroup"])
y = df["Crystal System"]

# Encode target
from sklearn.preprocessing import LabelEncoder
y = LabelEncoder().fit_transform(y)

# Fix bandstructure
if "Has Bandstructure" in X.columns:
    X["Has Bandstructure"] = X["Has Bandstructure"].replace({"True":1,"False":0})

# -----------------------------
# PIPELINE
# -----------------------------
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(random_state=42))
])

# -----------------------------
# HYPERPARAMETERS
# -----------------------------
param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [None, "sqrt", "log2"]
}

# -----------------------------
# K-FOLD
# -----------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# GRID SEARCH
# -----------------------------
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=kfold,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y)

# -----------------------------
# RESULTS
# -----------------------------
print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid.best_score_)

Best Parameters:
{'model__criterion': 'gini', 'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}

Best Cross-Validation Accuracy:
0.5902985074626865


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("lithium-ion batteries.csv")

# -----------------------------
# Remove leakage-prone and non-predictive columns
# -----------------------------
drop_cols = ["Crystal System", "Materials Id", "Formula", "Spacegroup"]
X = df.drop(columns=drop_cols, errors="ignore").copy()
y = df["Crystal System"].copy()

# Convert Has Bandstructure if present
if "Has Bandstructure" in X.columns:
    X["Has Bandstructure"] = X["Has Bandstructure"].replace(
        {"TRUE": 1, "FALSE": 0, "True": 1, "False": 0, True: 1, False: 0}
    ).astype(int)

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# -----------------------------
# CV setup
# -----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# Define models and parameter grids
# -----------------------------
models_and_params = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, random_state=42))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["lbfgs"],
            "model__class_weight": [None, "balanced"]
        }
    },

    "Decision Tree": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        "params": {
            "model__criterion": ["gini", "entropy"],
            "model__max_depth": [3, 5, 10, 15, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__class_weight": [None, "balanced"]
        }
    },

    "Random Forest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=42))
        ]),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [5, 10, 20, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__class_weight": [None, "balanced"]
        }
    },

    "K-Nearest Neighbor": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9],
            "model__weights": ["uniform", "distance"],
            "model__metric": ["minkowski", "euclidean", "manhattan"]
        }
    },

    "Support Vector Machine": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, random_state=42))
        ]),
        "params": {
            "model__C": [0.1, 1, 10],
            "model__kernel": ["linear", "rbf"],
            "model__gamma": ["scale", "auto"],
            "model__class_weight": [None, "balanced"]
        }
    },

    "Gradient Boosting": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(random_state=42))
        ]),
        "params": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5],
            "model__subsample": [0.8, 1.0]
        }
    },

    "XGBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=42,
                verbosity=0
            ))
        ]),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [3, 5, 7],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0]
        }
    },

    "Naive Bayes": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GaussianNB())
        ]),
        "params": {
            "model__var_smoothing": [1e-11, 1e-10, 1e-9, 1e-8]
        }
    },

    "AdaBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", AdaBoostClassifier(random_state=42))
        ]),
        "params": {
            "model__n_estimators": [50, 100, 200],
            "model__learning_rate": [0.01, 0.05, 0.1, 1.0]
        }
    },

    "CatBoost": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostClassifier(
                loss_function="MultiClass",
                verbose=0,
                random_state=42
            ))
        ]),
        "params": {
            "model__iterations": [100, 200, 300],
            "model__depth": [4, 6, 8],
            "model__learning_rate": [0.01, 0.05, 0.1]
        }
    }
}

# -----------------------------
# Tune models
# -----------------------------
best_models = {}
best_params = {}
summary_rows = []

for model_name, config in models_and_params.items():
    print(f"\nTuning {model_name} ...")

    grid = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X, y)

    best_models[model_name] = grid.best_estimator_
    best_params[model_name] = grid.best_params_

    print(f"Best params for {model_name}: {grid.best_params_}")
    print(f"Best CV accuracy for {model_name}: {grid.best_score_:.4f}")

# -----------------------------
# Evaluate tuned models
# -----------------------------
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc_ovr"
}

for model_name, model in best_models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)

    summary_rows.append({
        "Model": model_name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC_AUC": scores["test_roc_auc"].mean()
    })

results_df = pd.DataFrame(summary_rows).sort_values(by="Accuracy", ascending=False)
print("\nFinal tuned model comparison:")
print(results_df.round(4))

# -----------------------------
# Save outputs
# -----------------------------
results_df.round(4).to_csv("tuned_model_results.csv", index=False)
pd.DataFrame(best_params).T.to_csv("best_hyperparameters.csv")

print("\nSaved files:")
print("- tuned_model_results.csv")
print("- best_hyperparameters.csv")


Tuning Logistic Regression ...
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/tmp/ipykernel_6687/4277801845.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X["Has Bandstructure"] = X["Has Bandstructure"].replace(


Best params for Logistic Regression: {'model__C': 0.1, 'model__class_weight': None, 'model__solver': 'lbfgs'}
Best CV accuracy for Logistic Regression: 0.4866

Tuning Decision Tree ...
Fitting 5 folds for each of 180 candidates, totalling 900 fits
Best params for Decision Tree: {'model__class_weight': None, 'model__criterion': 'gini', 'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}
Best CV accuracy for Decision Tree: 0.5903

Tuning Random Forest ...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best params for Random Forest: {'model__class_weight': 'balanced', 'model__max_depth': 20, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 300}
Best CV accuracy for Random Forest: 0.6339

Tuning K-Nearest Neighbor ...
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best params for K-Nearest Neighbor: {'model__metric': 'manhattan', 'model__n_neighbors': 5, 'model__weights': 'distance'}
Best CV

In [ ]:
import re
import numpy as np
import pandas as pd

def parse_formula(formula):
    elements = re.findall(r'([A-Z][a-z]?)(\d*)', str(formula))
    counts = {}
    for el, num in elements:
        counts[el] = counts.get(el, 0) + (int(num) if num else 1)
    return counts

df = pd.read_csv("lithium-ion batteries.csv")

for el in ["Li", "Mn", "Fe", "Co", "Si", "O"]:
    df[el] = df["Formula"].apply(lambda x: parse_formula(x).get(el, 0))

df["TM"] = df[["Mn", "Fe", "Co"]].sum(axis=1)

df["Li_TM_ratio"] = df["Li"] / (df["TM"] + 1e-5)
df["O_Si_ratio"] = df["O"] / (df["Si"] + 1e-5)

In [ ]:
important_features = [
    "Formation Energy (eV)",
    "E Above Hull (eV)",
    "Band Gap (eV)",
    "Nsites",
    "Density (gm/cc)",
    "Volume",
    "Li", "Mn", "Fe", "Co", "Si", "O",
    "TM", "Li_TM_ratio", "O_Si_ratio"
]

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Define X and y for this specific workflow
X = df[important_features].copy()

# Convert 'Has Bandstructure' if it's in X (and not already converted by previous feature engineering)
# This is a precaution, as previous cells might have handled it, but ensuring it here for consistency.
if "Has Bandstructure" in X.columns:
    X["Has Bandstructure"] = X["Has Bandstructure"].replace({"True":1,"False":0, True:1, False:0}).astype(float)

y = LabelEncoder().fit_transform(df["Crystal System"])

pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("smote", SMOTE()),
    ("model", model)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X, y, cv=cv)

print("Accuracy:", scores.mean())

Accuracy: 0.6694029850746268


In [ ]:
model = XGBClassifier(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    min_child_weight=3,
    random_state=42
)

In [ ]:
StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

StratifiedKFold(n_splits=10, random_state=42, shuffle=True)

In [ ]:
df["Density_Volume"] = df["Density (gm/cc)"] * df["Volume"]
df["Energy_Gap"] = df["Formation Energy (eV)"] * df["Band Gap (eV)"]

In [ ]:
drop = ["Has Bandstructure"]

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv("lithium-ion batteries.csv")

# -----------------------------
# Remove leakage columns
# -----------------------------
X = df.drop(columns=["Crystal System", "Materials Id", "Formula", "Spacegroup"])
y = df["Crystal System"]

# Encode target
y = LabelEncoder().fit_transform(y)

# Fix bandstructure
if "Has Bandstructure" in X.columns:
    X["Has Bandstructure"] = X["Has Bandstructure"].replace({"True":1,"False":0})

# -----------------------------
# Model
# -----------------------------
model = XGBClassifier(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    min_child_weight=3,
    random_state=42,
    eval_metric='mlogloss'
)

# -----------------------------
# Pipeline
# -----------------------------
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", model)
])

# -----------------------------
# K-FOLD
# -----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# Metrics
# -----------------------------
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc_ovr"
}

# -----------------------------
# Cross-validation
# -----------------------------
scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)

# -----------------------------
# FINAL RESULTS
# -----------------------------
accuracy = np.mean(scores["test_accuracy"])
precision = np.mean(scores["test_precision"])
recall = np.mean(scores["test_recall"])
f1 = np.mean(scores["test_f1"])
roc_auc = np.mean(scores["test_roc_auc"])

print("\nFINAL RESULTS (5-FOLD CV)")
print("--------------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


FINAL RESULTS (5-FOLD CV)
--------------------------------
Accuracy : 0.5691
Precision: 0.5714
Recall   : 0.5478
F1-score : 0.5520
ROC-AUC  : 0.7659
